import importlib
import spark_data_check
importlib.reload(spark_data_check)

from spark_data_check import SparkDataCheck
# Python file for data checking in Spark. 
#### Name: Cole Hammett
#### Date: 27th of March, 2026
#### Purpose: This file contains functions for data checks on Spark DataFrames (null values, duplicates, and data types). 

# Part I: `SparkDataCheck` Example Usage

## Setup — Spark Session

In [1]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

spark = SparkSession.builder \
    .appName("SparkDataCheckDemo") \
    .getOrCreate()

print("Spark version:", spark.version)

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/03/27 03:34:03 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/03/27 03:34:04 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.


Spark version: 4.0.2


## Import and Reload the Module

In [2]:
import importlib
import spark_data_check
import pandas

importlib.reload(spark_data_check)
from spark_data_check import SparkDataCheck 

print("SparkDataCheck imported successfully.")

SparkDataCheck imported successfully.


## 1. Load Air Quality Data via `from_csv`

In [3]:
import pandas as pd

AIR_URL = "https://www4.stat.ncsu.edu/online/datasets/air.csv"

# Step 1: Download with pandas (handles HTTPS fine)
pdf = pd.read_csv(AIR_URL)

# Step 2: Create SparkDataCheck from the pandas DataFrame
sdc = SparkDataCheck.from_pandas(spark, pdf)

# Verify
print("Type:", type(sdc))
print("Schema:")
sdc.df.printSchema()
sdc.df.show(5)

Type: <class 'spark_data_check.SparkDataCheck'>
Schema:
root
 |-- Unnamed: 0: long (nullable = true)
 |-- Date: string (nullable = true)
 |-- Time: string (nullable = true)
 |-- CO(GT): double (nullable = true)
 |-- PT08.S1(CO): long (nullable = true)
 |-- NMHC(GT): long (nullable = true)
 |-- C6H6(GT): double (nullable = true)
 |-- PT08.S2(NMHC): long (nullable = true)
 |-- NOx(GT): long (nullable = true)
 |-- PT08.S3(NOx): long (nullable = true)
 |-- NO2(GT): long (nullable = true)
 |-- PT08.S4(NO2): long (nullable = true)
 |-- PT08.S5(O3): long (nullable = true)
 |-- T: double (nullable = true)
 |-- RH: double (nullable = true)
 |-- AH: double (nullable = true)
 |-- Season: string (nullable = false)



+----------+---------+--------+------+-----------+--------+--------+-------------+-------+------------+-------+------------+-----------+----+----+------+------+
|Unnamed: 0|     Date|    Time|CO(GT)|PT08.S1(CO)|NMHC(GT)|C6H6(GT)|PT08.S2(NMHC)|NOx(GT)|PT08.S3(NOx)|NO2(GT)|PT08.S4(NO2)|PT08.S5(O3)|   T|  RH|    AH|Season|
+----------+---------+--------+------+-----------+--------+--------+-------------+-------+------------+-------+------------+-----------+----+----+------+------+
|         0|3/10/2004|18:00:00|   2.6|       1360|     150|    11.9|         1046|    166|        1056|    113|        1692|       1268|13.6|48.9|0.7578|Spring|
|         1|3/10/2004|19:00:00|   2.0|       1292|     112|     9.4|          955|    103|        1174|     92|        1559|        972|13.3|47.7|0.7255|Spring|
|         2|3/10/2004|20:00:00|   2.2|       1402|      88|     9.0|          939|    131|        1140|    114|        1555|       1074|11.9|54.0|0.7502|Spring|
|         3|3/10/2004|21:00:00|   

### String Columns for Later 

In [4]:
# Step 1: Extract Month from the Date column
sdc.df = sdc.df.withColumn(
    "Month",
    F.month(F.to_date(F.col("Date"), "d/MM/yyyy"))
)

# Step 2: Now assign Season using the new Month column
sdc.df = sdc.df.withColumn(
    "Season",
    F.when(F.col("Month") == 5, "Spring")
     .when(F.col("Month").isin([6, 7, 8]), "Summer")
     .otherwise("Fall")
)

# Step 3: Add TempCat (note: your temp column is "T", not "Temp")
sdc.df = sdc.df.withColumn(
    "TempCat",
    F.when(F.col("T") >= 80, "Hot").otherwise("Cool")
)

print("Updated schema after adding engineered columns:")
sdc.df.printSchema()
sdc.df.show(8)

Updated schema after adding engineered columns:
root
 |-- Unnamed: 0: long (nullable = true)
 |-- Date: string (nullable = true)
 |-- Time: string (nullable = true)
 |-- CO(GT): double (nullable = true)
 |-- PT08.S1(CO): long (nullable = true)
 |-- NMHC(GT): long (nullable = true)
 |-- C6H6(GT): double (nullable = true)
 |-- PT08.S2(NMHC): long (nullable = true)
 |-- NOx(GT): long (nullable = true)
 |-- PT08.S3(NOx): long (nullable = true)
 |-- NO2(GT): long (nullable = true)
 |-- PT08.S4(NO2): long (nullable = true)
 |-- PT08.S5(O3): long (nullable = true)
 |-- T: double (nullable = true)
 |-- RH: double (nullable = true)
 |-- AH: double (nullable = true)
 |-- Season: string (nullable = false)
 |-- Month: integer (nullable = true)
 |-- TempCat: string (nullable = false)

+----------+---------+--------+------+-----------+--------+--------+-------------+-------+------------+-------+------------+-----------+----+----+------+------+-----+-------+
|Unnamed: 0|     Date|    Time|CO(GT)|PT08

### Helper

In [5]:
def fresh_sdc():
    """Return a clean SparkDataCheck built from the raw pandas DataFrame."""
    return SparkDataCheck.from_pandas(spark, pdf)

def fresh_sdc_with_strings():
    """Return a SparkDataCheck with Month, Season, and TempCat columns added."""
    obj = SparkDataCheck.from_pandas(spark, pdf)
    obj.df = obj.df.withColumn(
        "Month", F.month(F.to_date(F.col("Date"), "d/MM/yyyy"))
    )
    obj.df = obj.df.withColumn(
        "Season",
        F.when(F.col("Month") == 5, "Spring")
         .when(F.col("Month").isin([6, 7, 8]), "Summer")
         .otherwise("Fall")
    )
    obj.df = obj.df.withColumn(
        "TempCat",
        F.when(F.col("T") >= 80, "Hot").otherwise("Cool")
    )
    return obj

print("Helper functions defined.")

Helper functions defined.


## 2. `check_numeric_range` — 5 Examples


In [6]:
# ---------- Example 1: Both lower AND upper bounds on CO(GT) ----------
# CO(GT) is carbon monoxide concentration in mg/m^3.
# Realistic hourly values should be roughly 0–15; anything outside that range
# is either a sensor error or the -200 sentinel.
sdc_nr1 = fresh_sdc()

sdc_nr1.check_numeric_range("CO(GT)", lower=0.0, upper=15.0)

print("CO(GT) in [0.0, 15.0] — False rows are likely -200 sentinel values:")
sdc_nr1.df.select("CO(GT)", "CO(GT)_in_range").show(10)

CO(GT) in [0.0, 15.0] — False rows are likely -200 sentinel values:
+------+---------------+
|CO(GT)|CO(GT)_in_range|
+------+---------------+
|   2.6|           true|
|   2.0|           true|
|   2.2|           true|
|   2.2|           true|
|   1.6|           true|
|   1.2|           true|
|   1.2|           true|
|   1.0|           true|
|   0.9|           true|
|   0.6|           true|
+------+---------------+
only showing top 10 rows


In [7]:
# ---------- Example 2: LOWER bound only on T (temperature, Celsius) ----------
# Temperature in this Italian dataset should never drop below -20°C.
# We only care about the lower bound — there is no meaningful upper limit to enforce.
sdc_nr2 = fresh_sdc()

sdc_nr2.check_numeric_range("T", lower=-20.0)   # no upper argument

print("T >= -20.0 °C (lower bound only):")
sdc_nr2.df.select("T", "T_in_range").show(5)

T >= -20.0 °C (lower bound only):
+----+----------+
|   T|T_in_range|
+----+----------+
|13.6|      true|
|13.3|      true|
|11.9|      true|
|11.0|      true|
|11.2|      true|
+----+----------+
only showing top 5 rows


In [8]:
# ---------- Example 3: UPPER bound only on RH (relative humidity, %) ----------
# Relative humidity is physically bounded at 100 %. We only supply upper here.
sdc_nr3 = fresh_sdc()

sdc_nr3.check_numeric_range("RH", upper=100.0)  # no lower argument

print("RH <= 100.0 % (upper bound only):")
sdc_nr3.df.select("RH", "RH_in_range").show(5)

RH <= 100.0 % (upper bound only):
+----+-----------+
|  RH|RH_in_range|
+----+-----------+
|48.9|       true|
|47.7|       true|
|54.0|       true|
|60.0|       true|
|59.6|       true|
+----+-----------+
only showing top 5 rows


In [9]:
# ---------- Example 4: Message prints — passing a STRING column (Season) ----------
# Season is a string column. check_numeric_range should detect that, print a
# warning message, and return self WITHOUT appending any column.
sdc_nr4 = fresh_sdc_with_strings()

cols_before = sdc_nr4.df.columns
print("Attempting check_numeric_range on string column 'Season':")
sdc_nr4.check_numeric_range("Season", lower=0, upper=10)

# Confirm no new column was added
print("\nColumns before:", cols_before)
print("Columns after: ", sdc_nr4.df.columns)

Attempting check_numeric_range on string column 'Season':
check_numeric_range: 'Season' has dtype 'string', which is not a supported numeric type {'int', 'long', 'float', 'integer', 'bigint', 'double'}. No column appended.

Columns before: ['Unnamed: 0', 'Date', 'Time', 'CO(GT)', 'PT08.S1(CO)', 'NMHC(GT)', 'C6H6(GT)', 'PT08.S2(NMHC)', 'NOx(GT)', 'PT08.S3(NOx)', 'NO2(GT)', 'PT08.S4(NO2)', 'PT08.S5(O3)', 'T', 'RH', 'AH', 'Season', 'Month', 'TempCat']
Columns after:  ['Unnamed: 0', 'Date', 'Time', 'CO(GT)', 'PT08.S1(CO)', 'NMHC(GT)', 'C6H6(GT)', 'PT08.S2(NMHC)', 'NOx(GT)', 'PT08.S3(NOx)', 'NO2(GT)', 'PT08.S4(NO2)', 'PT08.S5(O3)', 'T', 'RH', 'AH', 'Season', 'Month', 'TempCat']


In [10]:
# ---------- Example 5: CHAINING two check_numeric_range calls ----------
# Methods return self, so calls can be chained. We check CO(GT) and T together,
# then call .df.show() once at the end to display both appended Boolean columns.
sdc_nr5 = fresh_sdc()

print("Chaining: CO(GT) in [0, 15] AND T in [-20, 45] — both columns appended:")
sdc_nr5 \
    .check_numeric_range("CO(GT)", lower=0.0, upper=15.0) \
    .check_numeric_range("T", lower=-20.0, upper=45.0) \
    .df.select("CO(GT)", "CO(GT)_in_range", "T", "T_in_range") \
    .show(10)

Chaining: CO(GT) in [0, 15] AND T in [-20, 45] — both columns appended:
+------+---------------+----+----------+
|CO(GT)|CO(GT)_in_range|   T|T_in_range|
+------+---------------+----+----------+
|   2.6|           true|13.6|      true|
|   2.0|           true|13.3|      true|
|   2.2|           true|11.9|      true|
|   2.2|           true|11.0|      true|
|   1.6|           true|11.2|      true|
|   1.2|           true|11.2|      true|
|   1.2|           true|11.3|      true|
|   1.0|           true|10.7|      true|
|   0.9|           true|10.7|      true|
|   0.6|           true|10.3|      true|
+------+---------------+----+----------+
only showing top 10 rows


## 3. `check_string_levels` — 5 Examples

In [11]:
# ---------- Example 1: All valid Season levels supplied — all rows should be True ----------
sdc_sl1 = fresh_sdc_with_strings()

print("Season in {Spring, Summer, Fall} — every row should return True:")
sdc_sl1.check_string_levels("Season", ["Spring", "Summer", "Fall"]) \
       .df.select("Season", "Season_in_levels").show(8)

Season in {Spring, Summer, Fall} — every row should return True:
+------+----------------+
|Season|Season_in_levels|
+------+----------------+
|  Fall|            true|
|  Fall|            true|
|  Fall|            true|
|  Fall|            true|
|  Fall|            true|
|  Fall|            true|
|  Fall|            true|
|  Fall|            true|
+------+----------------+
only showing top 8 rows


In [12]:
# ---------- Example 2: Only a SUBSET of levels — Spring and Fall rows flag False ----------
# Restricting to just ["Summer"] means only observations from June, July, and August
# will be flagged True; all other months return False.
sdc_sl2 = fresh_sdc_with_strings()

print("Season in {Summer} only — Spring and Fall flagged as False:")
sdc_sl2.check_string_levels("Season", ["Summer"]) \
       .df.select("Month", "Season", "Season_in_levels").show(10)

Season in {Summer} only — Spring and Fall flagged as False:
+-----+------+----------------+
|Month|Season|Season_in_levels|
+-----+------+----------------+
|   10|  Fall|           false|
|   10|  Fall|           false|
|   10|  Fall|           false|
|   10|  Fall|           false|
|   10|  Fall|           false|
|   10|  Fall|           false|
|   11|  Fall|           false|
|   11|  Fall|           false|
|   11|  Fall|           false|
|   11|  Fall|           false|
+-----+------+----------------+
only showing top 10 rows


In [13]:
# ---------- Example 3: Message prints — passing a NUMERIC column (CO(GT)) ----------
# CO(GT) is a double column. check_string_levels should detect that and print a
# warning, leaving the DataFrame unchanged.
sdc_sl3 = fresh_sdc_with_strings()

cols_before = sdc_sl3.df.columns
print("Attempting check_string_levels on numeric column 'CO(GT)':")
sdc_sl3.check_string_levels("CO(GT)", ["1.0", "2.0", "3.0"])

print("\nColumns unchanged:", sdc_sl3.df.columns == cols_before)

Attempting check_string_levels on numeric column 'CO(GT)':
Column 'CO(GT)' is of type 'double', not string. No check performed.

Columns unchanged: True


In [14]:
# ---------- Example 4: TempCat — only 'Cool' is in the levels set ----------
# Since T is in Celsius and never reaches 80°C in this dataset, TempCat is always
# "Cool". Checking ["Cool"] should therefore return True for every row.
sdc_sl4 = fresh_sdc_with_strings()

print("TempCat in {Cool} — all rows should be True (no 'Hot' readings exist):")
sdc_sl4.check_string_levels("TempCat", ["Cool"]) \
       .df.select("T", "TempCat", "TempCat_in_levels").show(5)

TempCat in {Cool} — all rows should be True (no 'Hot' readings exist):
+----+-------+-----------------+
|   T|TempCat|TempCat_in_levels|
+----+-------+-----------------+
|13.6|   Cool|             true|
|13.3|   Cool|             true|
|11.9|   Cool|             true|
|11.0|   Cool|             true|
|11.2|   Cool|             true|
+----+-------+-----------------+
only showing top 5 rows


In [15]:
# ---------- Example 5: CHAINING two check_string_levels calls ----------
# Check Season and TempCat in one chained expression, then call .df.show() once.
sdc_sl5 = fresh_sdc_with_strings()

print("Chaining: Season in {Summer} AND TempCat in {Cool, Hot}:")
sdc_sl5 \
    .check_string_levels("Season", ["Summer"]) \
    .check_string_levels("TempCat", ["Cool", "Hot"]) \
    .df.select("Month", "Season", "Season_in_levels", "TempCat", "TempCat_in_levels") \
    .show(8)

Chaining: Season in {Summer} AND TempCat in {Cool, Hot}:
+-----+------+----------------+-------+-----------------+
|Month|Season|Season_in_levels|TempCat|TempCat_in_levels|
+-----+------+----------------+-------+-----------------+
|   10|  Fall|           false|   Cool|             true|
|   10|  Fall|           false|   Cool|             true|
|   10|  Fall|           false|   Cool|             true|
|   10|  Fall|           false|   Cool|             true|
|   10|  Fall|           false|   Cool|             true|
|   10|  Fall|           false|   Cool|             true|
|   11|  Fall|           false|   Cool|             true|
|   11|  Fall|           false|   Cool|             true|
+-----+------+----------------+-------+-----------------+
only showing top 8 rows


## 4. `check_missing` — 5 Examples

In [16]:
# ---------- Example 1: Check CO(GT) for NULLs ----------
# The original dataset has some fully blank rows at the end of the Excel export;
# those parse as NULL when loaded through pandas → Spark.
sdc_cm1 = fresh_sdc()

print("check_missing on 'CO(GT)' — True where value is NULL:")
sdc_cm1.check_missing("CO(GT)").df.select("CO(GT)", "CO(GT)_missing").show(10)

check_missing on 'CO(GT)' — True where value is NULL:
+------+--------------+
|CO(GT)|CO(GT)_missing|
+------+--------------+
|   2.6|         false|
|   2.0|         false|
|   2.2|         false|
|   2.2|         false|
|   1.6|         false|
|   1.2|         false|
|   1.2|         false|
|   1.0|         false|
|   0.9|         false|
|   0.6|         false|
+------+--------------+
only showing top 10 rows


In [17]:
# ---------- Example 2: Check Date for NULLs ----------
# Date is a string column — check_missing works on any dtype.
# The trailing blank rows in the export show up here as NULL dates.
sdc_cm2 = fresh_sdc()

print("check_missing on 'Date' — string column, NULL where row is blank:")
sdc_cm2.check_missing("Date") \
       .df.select("Date", "Date_missing").show(10)

check_missing on 'Date' — string column, NULL where row is blank:
+---------+------------+
|     Date|Date_missing|
+---------+------------+
|3/10/2004|       false|
|3/10/2004|       false|
|3/10/2004|       false|
|3/10/2004|       false|
|3/10/2004|       false|
|3/10/2004|       false|
|3/11/2004|       false|
|3/11/2004|       false|
|3/11/2004|       false|
|3/11/2004|       false|
+---------+------------+
only showing top 10 rows


In [18]:
# ---------- Example 3: Check T (temperature) for NULLs ----------
sdc_cm3 = fresh_sdc()

print("check_missing on 'T' (temperature in Celsius):")
sdc_cm3.check_missing("T") \
       .df.select("T", "T_missing").show(5)

check_missing on 'T' (temperature in Celsius):
+----+---------+
|   T|T_missing|
+----+---------+
|13.6|    false|
|13.3|    false|
|11.9|    false|
|11.0|    false|
|11.2|    false|
+----+---------+
only showing top 5 rows


In [19]:
# ---------- Example 4: CHAINING check_missing on two columns ----------
# Chain CO(GT) and Date missing checks so both appended columns appear together.
sdc_cm4 = fresh_sdc()

print("Chaining check_missing on CO(GT) and Date:")
sdc_cm4 \
    .check_missing("CO(GT)") \
    .check_missing("Date") \
    .df.select("CO(GT)", "CO(GT)_missing", "Date", "Date_missing") \
    .show(10)

Chaining check_missing on CO(GT) and Date:
+------+--------------+---------+------------+
|CO(GT)|CO(GT)_missing|     Date|Date_missing|
+------+--------------+---------+------------+
|   2.6|         false|3/10/2004|       false|
|   2.0|         false|3/10/2004|       false|
|   2.2|         false|3/10/2004|       false|
|   2.2|         false|3/10/2004|       false|
|   1.6|         false|3/10/2004|       false|
|   1.2|         false|3/10/2004|       false|
|   1.2|         false|3/11/2004|       false|
|   1.0|         false|3/11/2004|       false|
|   0.9|         false|3/11/2004|       false|
|   0.6|         false|3/11/2004|       false|
+------+--------------+---------+------------+
only showing top 10 rows


In [20]:
# ---------- Example 5: Chain check_missing WITH check_numeric_range ----------
# Mixed-type chaining: first flag missing values in CO(GT), then flag out-of-range
# values in RH, all in one expression ending with .df.show().
sdc_cm5 = fresh_sdc()

print("Chain: check_missing('CO(GT)') → check_numeric_range('RH', lower=0, upper=100):")
sdc_cm5 \
    .check_missing("CO(GT)") \
    .check_numeric_range("RH", lower=0.0, upper=100.0) \
    .df.select("CO(GT)", "CO(GT)_missing", "RH", "RH_in_range") \
    .show(10)

Chain: check_missing('CO(GT)') → check_numeric_range('RH', lower=0, upper=100):
+------+--------------+----+-----------+
|CO(GT)|CO(GT)_missing|  RH|RH_in_range|
+------+--------------+----+-----------+
|   2.6|         false|48.9|       true|
|   2.0|         false|47.7|       true|
|   2.2|         false|54.0|       true|
|   2.2|         false|60.0|       true|
|   1.6|         false|59.6|       true|
|   1.2|         false|59.2|       true|
|   1.2|         false|56.8|       true|
|   1.0|         false|60.0|       true|
|   0.9|         false|59.7|       true|
|   0.6|         false|60.2|       true|
+------+--------------+----+-----------+
only showing top 10 rows


## 5. `numeric_summary` — 5 Examples

In [21]:
# ---------- Example 1: Single numeric column, no grouping (T — temperature) ----------
# numeric_summary returns a pandas DataFrame showing min and max of the column.
# Passing only col_name with no group_by gives a one-row global summary.
sdc_ns1 = fresh_sdc()

summary_t = sdc_ns1.numeric_summary(col_name="T")
print("Global min / max of T (°C):")
print(summary_t)

In [22]:
# ---------- Example 2: Single numeric column, no grouping (CO(GT)) ----------
# CO(GT) contains -200 sentinel values for missing readings.
# The min should surface those sentinels, confirming the data needs cleaning.
sdc_ns2 = fresh_sdc()

summary_co = sdc_ns2.numeric_summary(col_name="CO(GT)")
print("Global min / max of CO(GT) — negative min exposes -200 sentinels:")
print(summary_co)